# Tutorial 8: Train NicheTrans on human lymph node data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_human_lymph_node import Lymph_node

from utils.utils import *
from utils.utils_training_human_lymph_node import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_human_lymph_node.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.1, n_source=3000, workers=4, adata_path='/mnt/datadisk0/Processed_DATA/2024_nm_human_lymph_nodes/', cell_type_visualize=False, cell_type_visualization_dir=None, cell_type_visualization_dpi=150, max_epoch=20, stepsize=10, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = Lymph_node(adata_path=args.adata_path, n_top_genes=args.n_source)
trainloader, testloader = human_node_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.protein_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,
                   n_spot_types=dataset.n_spot_types)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

  Slice slice1: Leiden found 6 local clusters (sizes: [982, 891, 599, 536, 343, 133])
  Slice slice2: Leiden found 4 local clusters (sizes: [1608, 1257, 371, 123])
  Reference slice slice2 defines the fixed global cell-type space with 4 classes
    Align slice=slice1 local_cluster=0 -> reference/global=0 cosine_similarity=0.9938
    Align slice=slice1 local_cluster=1 -> reference/global=1 cosine_similarity=0.9920
    Align slice=slice1 local_cluster=2 -> reference/global=0 cosine_similarity=0.9647
    Align slice=slice1 local_cluster=3 -> reference/global=1 cosine_similarity=0.9670
    Align slice=slice1 local_cluster=4 -> reference/global=2 cosine_similarity=0.9647
    Align slice=slice1 local_cluster=5 -> reference/global=2 cosine_similarity=0.9071
  Slice slice1: mapped 6/6 local clusters into the slice2-defined global space
  Cross-slice alignment complete: 4 global cell types
------Calculating spatial graph...
The graph contains 13638 edges, 3484 cells.
3.9145 neighbors per cell o

### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################
    
pearson = test(model, testloader, device=device)
torch.save(model.state_dict(), 'NicheTrans_human_lymph_node_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 108/108	 Loss 0.776017 (0.969896)
==> Epoch 2/20
Batch 108/108	 Loss 0.463646 (0.880194)
==> Epoch 3/20
Batch 108/108	 Loss 0.689075 (0.849379)
==> Epoch 4/20
Batch 108/108	 Loss 0.376359 (0.823226)
==> Epoch 5/20
Batch 108/108	 Loss 0.979148 (0.799855)
==> Epoch 6/20
Batch 108/108	 Loss 0.433995 (0.792620)
==> Epoch 7/20
Batch 108/108	 Loss 0.749358 (0.779125)
==> Epoch 8/20
Batch 108/108	 Loss 0.492908 (0.761805)
==> Epoch 9/20
Batch 108/108	 Loss 0.545333 (0.743899)
==> Epoch 10/20
Batch 108/108	 Loss 0.678788 (0.718315)
==> Epoch 11/20
Batch 108/108	 Loss 1.393925 (0.741864)
==> Epoch 12/20
Batch 108/108	 Loss 0.363408 (0.700594)
==> Epoch 13/20
Batch 108/108	 Loss 1.808725 (0.689572)
==> Epoch 14/20
Batch 108/108	 Loss 0.484740 (0.663999)
==> Epoch 15/20
Batch 108/108	 Loss 0.513999 (0.685101)
==> Epoch 16/20
Batch 108/108	 Loss 0.389751 (0.668603)
==> Epoch 17/20
Batch 108/108	 Loss 0.613775 (0.673185)
==> Epoch 18/20
Batch 108/108	 Loss 1.050252 (0.662878)
=